<a href="https://colab.research.google.com/github/vunhankhanhmcs3-cmd/master-thesis-mooc-recommendation/blob/main/04_RL_Training_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os
from collections import defaultdict
from tqdm import tqdm
import random
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive

In [ ]:
# 1. Cấu hình
drive.mount('/content/drive')

# Đường dẫn (Cần đảm bảo bạn đã upload file raw vào đây)
BASE_PATH = '/content/drive/MyDrive/01. THESIS/My suggestion/'
PROCESSED_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/processed/')
RELATIONS_PATH = os.path.join(BASE_PATH, 'data 07.11.2026/raw/relations/')
if not os.path.exists(PROCESSED_PATH): os.makedirs(PROCESSED_PATH)

Mounted at /content/drive


In [ ]:
# 1. Định nghĩa Môi trường Đồ thị tri thức Giáo dục (MOOC Environment)
class MOOCEnvironment:
    def __init__(self, data_path):
        # Tải không gian nhúng tiền huấn luyện (thừa hưởng từ file 03)
        self.entity_emb = np.load(os.path.join(data_path, 'entity_embeddings.npy'))
        self.relation_emb = np.load(os.path.join(data_path, 'relation_embeddings.npy'))

        # Tải cấu trúc Đồ thị tri thức sạch (xây từ tập Train)
        kg_df = pd.read_csv(os.path.join(data_path, 'kg_final.txt'), sep=' ', header=None)
        self.graph = defaultdict(list)
        for _, row in kg_df.iterrows():
            self.graph[int(row[0])].append((int(row[1]), int(row[2])))

        # Đọc thông số thống kê để xác định ranh giới ID của Khóa học
        with open(os.path.join(data_path, 'entity_stats.txt'), 'r') as f:
            stats = {line.split('=')[0]: int(line.split('=')[1]) for line in f}
        self.course_start_id = stats['user_num']
        self.course_end_id = stats['user_num'] + stats['course_num']
        self.visited_nodes = set()

    def reset(self, user_id, target_course=None):
        self.state = user_id
        self.target_course = target_course
        self.path = [(user_id, -1)]
        self.visited_nodes = {user_id}
        return self.get_state()

    def get_state(self):
        user_id = self.path[0][0]
        # Vector trạng thái là sự kết hợp (nối chuỗi) giữa nhúng Người học mục tiêu và thực thể hiện tại
        return np.concatenate([self.entity_emb[user_id], self.entity_emb[self.state]])

    def get_valid_actions(self):
        actions = self.graph[self.state]
        # Kiểm soát chu trình: Chặn tác nhân quay lui lại các nút đã đi qua trên lộ trình
        return [(r, n) for r, n in actions if n not in self.visited_nodes]

    def step(self, action):
        r, next_node = action
        self.state = next_node
        self.path.append((next_node, r))
        self.visited_nodes.add(next_node)

        reward = 0.0
        # GIỚI HẠN CỨNG: MAX_STEPS = 3 (Tương ứng chuỗi 2 chặng nhảy để khớp với file 05)
        done = len(self.path) >= 4 # 3 bước đi (cạnh) + 1 nút gốc ban đầu

        # === CƠ CHẾ HÀM PHẦN THƯỞNG CẢI TIẾN ===
        if self.target_course:
            # 1. Phần thưởng mục tiêu chặng cuối (R_target)
            if next_node == self.target_course:
                reward += 10.0
                done = True

            # 2. Phần thưởng ngữ nghĩa dẫn đường hình học (R_semantic)
            curr_vec = self.entity_emb[next_node].reshape(1, -1)
            target_vec = self.entity_emb[self.target_course].reshape(1, -1)
            sim = cosine_similarity(curr_vec, target_vec)[0][0]
            if sim > 0:
                reward += 1.0 * sim

        # 3. Phần thưởng sư phạm định hướng thực thể tri thức (R_pedagogical)
        # Khuyến khích tác nhân đi qua các quan hệ Khái niệm (Concept) - Relation ID = 2
        if r == 2:
            reward += 0.5

        return self.get_state(), reward, done, self.path

In [ ]:
# 2. Định nghĩa Mạng chính sách dựa trên Perceptron Đa lớp (Policy Network MLP)
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        # Kiến trúc 2 tầng tuyến tính tối ưu tham số để chạy mượt trên Colab T4 GPU
        self.l1 = nn.Linear(state_dim + action_dim, 128)
        self.l2 = nn.Linear(128, 1)

    def forward(self, state, action_emb):
        num_actions = action_emb.size(1)
        state_exp = state.unsqueeze(1).expand(-1, num_actions, -1)
        x = torch.cat([state_exp, action_emb], dim=-1)
        x = F.elu(self.l1(x)) # Hàm kích hoạt ELU theo cấu trúc thiết kế luận văn
        return self.l2(x).squeeze(-1)

    def select_action(self, state, valid_actions, ent_emb, rel_emb):
        act_vecs = []
        for r, n in valid_actions:
            act_vecs.append(rel_emb[r] + ent_emb[n])

        act_tensor = torch.tensor(np.array(act_vecs), dtype=torch.float32).unsqueeze(0)
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)

        # Tính toán phân phối Softmax để lấy mẫu hành động ngẫu nhiên
        probs = F.softmax(self.forward(state_tensor, act_tensor), dim=1)
        action_idx = torch.multinomial(probs, 1).item()
        return action_idx, torch.log(probs[0, action_idx])

In [ ]:
# 3. Chuẩn bị Dữ liệu Huấn luyện sạch (Đọc từ sản phẩm của bước 8:1:1)
env = MOOCEnvironment(PROCESSED_PATH)
policy_net = PolicyNetwork(200, 100) # State_dim = 200, Action_dim = 100
optimizer = optim.Adam(policy_net.parameters(), lr=0.0005)

# ĐỌC FILE TRAIN.TXT (Sản phẩm phân chia 8:1:1 nghiêm ngặt từ File 02)
train_df = pd.read_csv(os.path.join(PROCESSED_PATH, 'train.txt'), sep=' ', header=None, names=['u', 'c', 'l'])

# Ánh xạ tịnh tiến ID cục bộ sang ID toàn cục trong KG
train_df['c'] = train_df['c'] + env.course_start_id
user_targets = train_df.groupby('u')['c'].apply(list).to_dict()
train_users = list(user_targets.keys())

print(f"✅ Đã nạp thành công {len(train_users)} người học từ tập Train để đưa vào pha huấn luyện RL.")

✅ Đã nạp thành công 6486 người học từ tập Train để đưa vào pha huấn luyện RL.


In [ ]:
# 4. Vòng lặp Huấn luyện Thuật toán REINFORCE Cải tiến (5.000 Episodes)
EPISODES = 5000
pbar = tqdm(range(EPISODES))
EPSILON_START, EPSILON_END, DECAY = 0.5, 0.05, 0.001 # Tham số Guided Exploration (Nhắc bài cho Agent)

for i_ep in pbar:
    # Bốc ngẫu nhiên một người dùng và một khóa học mục tiêu tương ứng TRONG TẬP TRAIN
    uid = random.choice(train_users)
    if not user_targets[uid]: continue
    target = random.choice(user_targets[uid])

    state = env.reset(uid, target)
    log_probs, rewards = [], []

    # Khai phá dẫn đường (Epsilon giảm dần theo thời gian)
    epsilon = EPSILON_END + (EPSILON_START - EPSILON_END) * np.exp(-DECAY * i_ep)

    for _ in range(3): # Chạy tối đa 3 bước đi (MAX_STEPS = 3)
        valid = env.get_valid_actions()
        if not valid: break

        # Cơ chế Guided Exploration: Đôi khi định hướng hình học cho Agent đi nhanh hơn
        if random.random() < epsilon:
            best_idx, best_sim = 0, -1
            tgt_vec = env.entity_emb[target]
            for idx, (_, n) in enumerate(valid):
                n_vec = env.entity_emb[n]
                sim = np.dot(n_vec, tgt_vec)
                if sim > best_sim: best_sim, best_idx = sim, idx

            _, lp = policy_net.select_action(state, valid, env.entity_emb, env.relation_emb)
            idx = best_idx
        else:
            # Tác nhân tự chọn hành động dựa trên phân phối xác suất mạng chính sách
            idx, lp = policy_net.select_action(state, valid, env.entity_emb, env.relation_emb)

        state, r, done, _ = env.step(valid[idx])
        log_probs.append(lp)
        rewards.append(r)
        if done: break

    # Cập nhật Gradient Chính sách (Policy Gradient Update)
    R = 0
    returns = []
    for r in rewards[::-1]:
        R = r + 0.99 * R
        returns.insert(0, R)
    returns = torch.tensor(returns, dtype=torch.float32)

    # Chuẩn hóa phần thưởng bằng độ lệch chuẩn trên lô huấn luyện để ổn định độ dốc
    if returns.std() > 0:
        returns = (returns - returns.mean()) / (returns.std() + 1e-9)

    loss = []
    for lp, R in zip(log_probs, returns):
        loss.append(-lp * R)

    optimizer.zero_grad()
    if loss:
        torch.stack(loss).sum().backward()
        optimizer.step()

    if i_ep % 100 == 0:
        pbar.set_description(f"Ep {i_ep} (Eps={epsilon:.2f})")

Ep 0 (Eps=0.50):   0%|          | 1/5000 [00:00<18:41,  4.46it/s]/tmp/ipykernel_2667/3262861512.py:51: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  if returns.std() > 0:
/tmp/ipykernel_2667/3262861512.py:51: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  if returns.std() > 0:
/tmp/ipykernel_2667/3262861512.py:51: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)
  if returns.std() > 0:
/tmp/ipykernel_2667/3262861512.py:51: UserWarning: std(): degrees of freedom 

In [ ]:
# 5. Lưu trọng số mô hình tốt nhất phục vụ pha Đánh giá (File 05)
model_save_path = os.path.join(PROCESSED_PATH, 'policy_net.pth')
torch.save(policy_net.state_dict(), model_save_path)

print(f"💾 Đã lưu trọng số mạng chính sách vào Drive: {model_save_path}")
print("🚀 Quá trình huấn luyện hoàn tất! Bước tiếp theo: Tiến hành đánh giá tại 05_RL_Evaluation.ipynb")

💾 Đã lưu trọng số mạng chính sách vào Drive: /content/drive/MyDrive/01. THESIS/My suggestion/data 07.11.2026/processed/policy_net.pth
🚀 Quá trình huấn luyện hoàn tất! Bước tiếp theo: Tiến hành đánh giá tại 05_RL_Evaluation.ipynb
